# IIT-8. La parité, de bout en bout (Lean)

## Navigation
**Série** : [IIT](./README.md) | **Partie** : Escalier Lean (marche 2/3)
**Précédent** : [IIT-7 : la coupe et le verdict](./IIT-07-Integration-Par-Coupes-Lean.ipynb) | **Suivant** : [IIT-9 : capstone intégré et trivial](./IIT-09-Capstone-Integre-et-Trivial-Lean.ipynb)


Cette marche expose les déclarations 11 à 16 de `IIT/Integration.lean` : la parité d'un état
(`parity`), le système qui la diffuse (`broadcast`), et les quatre lemmes qui la calculent
(`parity_allFalse`, `parity_allTrue`, `parity_oneHot_zero`, `parity_const`).

L'objectif n'est pas d'admirer des preuves : la parité est **calculée** sur des états concrets
(`n = 3`, `n = 4`, tout-faux, tout-vrai, bit unique), et chaque lemme devient un fait vérifiable à
la main avant d'être invoqué. C'est ce qui rendra la marche 3 lisible : sans elle, `broadcast`
tomberait du ciel.


## 1. Environnement — charger le lake


In [1]:
import IIT.Integration
#eval 2 + 2
#check @IIT.parity
#check @IIT.broadcast


import IIT.Integration
#eval 2 + 2
─────▶  4
#check @IIT.parity
──────▶  IIT.parity : (n : Nat) → IIT.State n → Bool
#check @IIT.broadcast
──────▶  @IIT.broadcast : {n : Nat} → IIT.System n

--% env 0

Raw input:
{"cmd": "import IIT.Integration\n#eval 2 + 2\n#check @IIT.parity\n#check @IIT.broadcast\n"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "4"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "IIT.parity : (n : Nat) → IIT.State n → Bool"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data": "@IIT.broadcast : {n : Nat} → IIT.System n"}],
 "env": 0}

**Interprétation.** `2 + 2` confirme que le kernel Lean tourne ; puis les deux signatures.
`parity` prend une taille `n` et un état, et rend un `Bool` ; `broadcast` prend implicitement `n`
et rend un `System n` — une fonction d'états, exactement comme les systèmes de la marche 1.


## 2. La définition — une récurrence qui additionne les bits

Le lake définit la parité par récurrence sur `n` :

```lean
def parity : (n : Nat) → State n → Bool
  | 0, _ => false
  | n + 1, s => xor (s ⟨0, Nat.succ_pos n⟩) (parity n (fun i => s i.succ))
```

Un état de taille `n + 1` se lit : son bit de tête (indice 0) puis sa queue (les indices 1 à n,
décalés par `Fin.succ`). La parité est le **xor** de la tête avec la parité de la queue — le cas
`n = 0` ferme le calcul à `false`.


In [2]:
-- La recurrence deployee sur l'etat [true, false, true, false] de n = 4 :
-- chaque etage isole le bit de tete, le cas n = 0 ferme a false.
#eval xor true (xor false (xor true (xor false false)))
-- Le lake, sur le meme etat ecrit en fonction :
#eval IIT.parity 4 (fun i => decide (i.val % 2 = 0))


-- La recurrence deployee sur l'etat [true, false, true, false] de n = 4 :
-- chaque etage isole le bit de tete, le cas n = 0 ferme a false.
#eval xor true (xor false (xor true (xor false false)))
─────▶  false
-- Le lake, sur le meme etat ecrit en fonction :
#eval IIT.parity 4 (fun i => decide (i.val % 2 = 0))
─────▶  false

--% env 1

Raw input:
{"cmd": "-- La recurrence deployee sur l'etat [true, false, true, false] de n = 4 :\n-- chaque etage isole le bit de tete, le cas n = 0 ferme a false.\n#eval xor true (xor false (xor true (xor false false)))\n-- Le lake, sur le meme etat ecrit en fonction :\n#eval IIT.parity 4 (fun i => decide (i.val % 2 = 0))\n", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "false"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "false"}],
 "env": 1}

**Interprétation.** Les deux lignes rendent `false` : la parité d'un état est le nombre de bits
vrais **modulo 2** — ici deux bits vrais, donc pair, donc `false`. La récurrence ne fait rien
d'autre qu'additionner tous les bits dans `xor`.


## 3. Tout-faux, tout-vrai — les deux premiers faits


In [3]:
-- Les faits d'abord, a la main, sur n = 3 puis n = 4.
#eval [IIT.parity 3 (fun _ => false), IIT.parity 4 (fun _ => false)]
#eval [IIT.parity 3 (fun _ => true),  IIT.parity 4 (fun _ => true)]
-- Le meme tableau, vu par % 2 : l'imparite de n.
#eval [decide (3 % 2 = 1), decide (4 % 2 = 1)]


-- Les faits d'abord, a la main, sur n = 3 puis n = 4.
#eval [IIT.parity 3 (fun _ => false), IIT.parity 4 (fun _ => false)]
─────▶  [false, false]
#eval [IIT.parity 3 (fun _ => true),  IIT.parity 4 (fun _ => true)]
─────▶  [true, false]
-- Le meme tableau, vu par % 2 : l'imparite de n.
#eval [decide (3 % 2 = 1), decide (4 % 2 = 1)]
─────▶  [true, false]

--% env 2

Raw input:
{"cmd": "-- Les faits d'abord, a la main, sur n = 3 puis n = 4.\n#eval [IIT.parity 3 (fun _ => false), IIT.parity 4 (fun _ => false)]\n#eval [IIT.parity 3 (fun _ => true),  IIT.parity 4 (fun _ => true)]\n-- Le meme tableau, vu par % 2 : l'imparite de n.\n#eval [decide (3 % 2 = 1), decide (4 % 2 = 1)]\n", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "[false, false]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "[true, false]"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "[true, false]"}],
 "env": 2}

In [4]:
-- Les lemmes du lake, qui tiennent ces faits pour TOUT n.
#check @IIT.parity_allFalse
#check @IIT.parity_allTrue
example : IIT.parity 4 (fun _ => true) = false := by
  rw [IIT.parity_allTrue]
  decide
#print axioms IIT.parity_allTrue


-- Les lemmes du lake, qui tiennent ces faits pour TOUT n.
#check @IIT.parity_allFalse
──────▶  IIT.parity_allFalse : ∀ (n : Nat) (s : IIT.State n), (∀ (i : Fin n), s i = false) → IIT.parity n s = false
#check @IIT.parity_allTrue
──────▶  @IIT.parity_allTrue : ∀ {n : Nat}, (IIT.parity n fun x => true) = decide (n % 2 = 1)
example : IIT.parity 4 (fun _ => true) = false := by
  rw [IIT.parity_allTrue]
  decide
#print axioms IIT.parity_allTrue
──────▶  'IIT.parity_allTrue' depends on axioms: [propext, Quot.sound]

--% env 3

Raw input:
{"cmd": "-- Les lemmes du lake, qui tiennent ces faits pour TOUT n.\n#check @IIT.parity_allFalse\n#check @IIT.parity_allTrue\nexample : IIT.parity 4 (fun _ => true) = false := by\n  rw [IIT.parity_allTrue]\n  decide\n#print axioms IIT.parity_allTrue\n", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "IIT.parity_allFalse : ∀ (n : Nat) (s : IIT.State n), (∀ (i : Fin n), s i = false) → IIT.parity n s = false"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@IIT.parity_allTrue : ∀ {n : Nat}, (IIT.parity n fun x => true) = decide (n % 2 = 1)"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data": "'IIT.parity_allTrue' depends on axioms: [propext, Quot.sound]"}],
 "env": 3}

**Interprétation.** Tout-faux rend `false` quelle que soit la taille ; tout-vrai rend `true`
pour `n = 3` et `false` pour `n = 4` — exactement `decide (n % 2 = 1)`, l'imparité de `n`. Le
tableau calculé à la main et les deux lemmes disent la même chose ; la différence est le
quantificateur : les `#eval` le vérifient sur des cas, `parity_allFalse` et `parity_allTrue` le
tiennent pour tout `n`. Et la preuve du lemme ne dépend d'aucun axiome.


## 4. Le bit unique — et pourquoi l'hypothèse `0 < n`

`parity_oneHot_zero` est le seul lemme de la famille qui **exige** une hypothèse : `0 < n`.
C'est l'occasion de voir pourquoi `n = 0` est un cas dégénéré du lake.


In [5]:
-- Un seul bit vrai, d'indice 0 : la parite est vraie, quelle que soit la taille.
#eval [IIT.parity 1 (fun j => decide (j.val = 0)),
       IIT.parity 3 (fun j => decide (j.val = 0)),
       IIT.parity 5 (fun j => decide (j.val = 0))]
#check @IIT.parity_oneHot_zero
#print axioms IIT.parity_oneHot_zero


-- Un seul bit vrai, d'indice 0 : la parite est vraie, quelle que soit la taille.
#eval [IIT.parity 1 (fun j => decide (j.val = 0)),
─────▶  [true, true, true]
       IIT.parity 3 (fun j => decide (j.val = 0)),
       IIT.parity 5 (fun j => decide (j.val = 0))]
#check @IIT.parity_oneHot_zero
──────▶  IIT.parity_oneHot_zero : ∀ (n : Nat), 0 < n → (IIT.parity n fun j => decide (↑j = 0)) = true
#print axioms IIT.parity_oneHot_zero
──────▶  'IIT.parity_oneHot_zero' depends on axioms: [propext]

--% env 4

Raw input:
{"cmd": "-- Un seul bit vrai, d'indice 0 : la parite est vraie, quelle que soit la taille.\n#eval [IIT.parity 1 (fun j => decide (j.val = 0)),\n       IIT.parity 3 (fun j => decide (j.val = 0)),\n       IIT.parity 5 (fun j => decide (j.val = 0))]\n#check @IIT.parity_oneHot_zero\n#print axioms IIT.parity_oneHot_zero\n", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "[true, true, true]"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "IIT.parity_oneHot_zero : ∀ (n : Nat), 0 < n → (IIT.parity n fun j => decide (↑j = 0)) = true"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data": "'IIT.parity_oneHot_zero' depends on axioms: [propext]"}],
 "env": 4}

In [6]:
-- n = 0 : l'etat vide n'a aucun bit a lire -- la parite y est false par definition,
-- pas par preuve.
#eval IIT.parity 0 (fun _ => false)
-- Et le vocabulaire de la marche 1 n'y existe pas : une coupe exige deux cotes NON VIDES,
-- impossibles a temoigner quand n = 0 (ha et hb exigent chacun un indice valide).
#check @IIT.Cut


-- n = 0 : l'etat vide n'a aucun bit a lire -- la parite y est false par definition,
-- pas par preuve.
#eval IIT.parity 0 (fun _ => false)
─────▶  false
-- Et le vocabulaire de la marche 1 n'y existe pas : une coupe exige deux cotes NON VIDES,
-- impossibles a temoigner quand n = 0 (ha et hb exigent chacun un indice valide).
#check @IIT.Cut
──────▶  IIT.Cut : Nat → Type

--% env 5

Raw input:
{"cmd": "-- n = 0 : l'etat vide n'a aucun bit a lire -- la parite y est false par definition,\n-- pas par preuve.\n#eval IIT.parity 0 (fun _ => false)\n-- Et le vocabulaire de la marche 1 n'y existe pas : une coupe exige deux cotes NON VIDES,\n-- impossibles a temoigner quand n = 0 (ha et hb exigent chacun un indice valide).\n#check @IIT.Cut\n", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "false"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data": "IIT.Cut : Nat → Type"}],
 "env": 5}

**Interprétation.** Pour `n = 1, 3, 5`, le bit unique rend la parité impaire — un seul bit vrai,
un compte impair. Le lemme le tient pour tout `n` **strictement positif** ; à `n = 0`, la
récurrence rend `false` par définition (aucun bit à lire) et surtout **aucune coupe n'existe** :
une `Cut` exige deux côtés non vides, et il n'y a aucun indice à témoigner. Le cas `n = 0` est
dégénéré pour toute la théorie du lake — c'est ce que l'hypothèse `0 < n` encode.


## 5. L'état constant — le lemme qui généralise


In [7]:
-- Un etat constant de bit b : la parite vaut b && imparite de n.
#eval [IIT.parity 5 (fun _ => true), IIT.parity 5 (fun _ => false)]
#eval [IIT.parity 6 (fun _ => true), IIT.parity 6 (fun _ => false)]
#check @IIT.parity_const
example : IIT.parity 6 (fun _ => true) = false := by
  rw [IIT.parity_const]
  decide


-- Un etat constant de bit b : la parite vaut b && imparite de n.
#eval [IIT.parity 5 (fun _ => true), IIT.parity 5 (fun _ => false)]
─────▶  [true, false]
#eval [IIT.parity 6 (fun _ => true), IIT.parity 6 (fun _ => false)]
─────▶  [false, false]
#check @IIT.parity_const
──────▶  IIT.parity_const : ∀ (n : Nat) (b : Bool), (IIT.parity n fun x => b) = (b && decide (n % 2 = 1))
example : IIT.parity 6 (fun _ => true) = false := by
  rw [IIT.parity_const]
  decide

--% env 6

Raw input:
{"cmd": "-- Un etat constant de bit b : la parite vaut b && imparite de n.\n#eval [IIT.parity 5 (fun _ => true), IIT.parity 5 (fun _ => false)]\n#eval [IIT.parity 6 (fun _ => true), IIT.parity 6 (fun _ => false)]\n#check @IIT.parity_const\nexample : IIT.parity 6 (fun _ => true) = false := by\n  rw [IIT.parity_const]\n  decide\n", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "[true, false]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "[false, false]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "IIT.parity_const : ∀ (n : Nat) (b : Bool), (IIT.parity n fun x => b) = (b && decide (n % 2 = 1))"}],
 "env": 6}

**Interprétation.** `parity_const` unifie les deux lemmes précédents : pour un état constant de
bit `b`, la parité vaut `b && decide (n % 2 = 1)`. Bit faux → toujours `false` (c'est
`parity_allFalse`) ; bit vrai → l'imparité de `n` (c'est `parity_allTrue`). Quatre déclarations
de la famille parité, un seul tableau à retenir.


## 6. `broadcast` — le système qui diffuse la parité

`broadcast` est le système dont l'état suivant est **uniformément** la parité de l'état courant :
chaque élément reçoit la même information globale. C'est le système que la marche 1 a laissé au
bord du verdict (`integrated`), et dont la marche 3 prouvera les deux théorèmes.


In [8]:
-- Les huit etats de n = 3, dans l'ordre binaire (comme en marche 1).
def huitEtats : List (IIT.State 3) :=
  [ fun _ => false,
    fun i => decide (i.val = 2),
    fun i => decide (i.val = 1),
    fun i => decide (i.val = 1 || i.val = 2),
    fun i => decide (i.val = 0),
    fun i => decide (i.val = 0 || i.val = 2),
    fun i => decide (i.val = 0 || i.val = 1),
    fun _ => true ]
#eval huitEtats.map (fun s => [s 0, s 1, s 2])
#eval huitEtats.map (fun s => IIT.parity 3 s)
#eval huitEtats.map (fun s => [IIT.broadcast s 0, IIT.broadcast s 1, IIT.broadcast s 2])


-- Les huit etats de n = 3, dans l'ordre binaire (comme en marche 1).
def huitEtats : List (IIT.State 3) :=
  [ fun _ => false,
    fun i => decide (i.val = 2),
    fun i => decide (i.val = 1),
    fun i => decide (i.val = 1 || i.val = 2),
    fun i => decide (i.val = 0),
    fun i => decide (i.val = 0 || i.val = 2),
    fun i => decide (i.val = 0 || i.val = 1),
    fun _ => true ]
#eval huitEtats.map (fun s => [s 0, s 1, s 2])
─────▶  [[false, false, false],
 [false, false, true],
 [false, true, false],
 [false, true, true],
 [true, false, false],
 [true, false, true],
 [true, true, false],
 [true, true, true]]
#eval huitEtats.map (fun s => IIT.parity 3 s)
─────▶  [false, true, true, false, true, false, false, true]
#eval huitEtats.map (fun s => [IIT.broadcast s 0, IIT.broadcast s 1, IIT.broadcast s 2])
─────▶  [[false, false, false],
 [true, true, true],
 [true, true, true],
 [false, false, false],
 [true, true, true],
 [false, false, false],
 [false, false, false],
 [true, true, true]]

--% env 7

Raw input:
{"cmd": "-- Les huit etats de n = 3, dans l'ordre binaire (comme en marche 1).\ndef huitEtats : List (IIT.State 3) :=\n  [ fun _ => false,\n    fun i => decide (i.val = 2),\n    fun i => decide (i.val = 1),\n    fun i => decide (i.val = 1 || i.val = 2),\n    fun i => decide (i.val = 0),\n    fun i => decide (i.val = 0 || i.val = 2),\n    fun i => decide (i.val = 0 || i.val = 1),\n    fun _ => true ]\n#eval huitEtats.map (fun s => [s 0, s 1, s 2])\n#eval huitEtats.map (fun s => IIT.parity 3 s)\n#eval huitEtats.map (fun s => [IIT.broadcast s 0, IIT.broadcast s 1, IIT.broadcast s 2])\n", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 5},
   "data":
   "[[false, false, false],\n [false, false, true],\n [false, true, false],\n [false, true, true],\n [true, false, false],\n [true, false, true],\n [true, true, false],\n [true, true, true]]"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 5},
   "data": "[false, true, true, false, true, false, false, true]"},
  {"severity": "info",
   "pos": {"line": 13, "column": 0},
   "endPos": {"line": 13, "column": 5},
   "data":
   "[[false, false, false],\n [true, true, true],\n [true, true, true],\n [false, false, false],\n [true, true, true],\n [false, false, false],\n [false, false, false],\n [true, true, true]]"}],
 "env": 7}

**Interprétation.** Huit états en entrée, et la troisième table ne contient plus que **deux
lignes différentes** : `[false, false, false]` quand la parité est fausse, `[true, true, true]`
quand elle est vraie. L'image de `broadcast` se réduit à deux états constants — l'information
globale est partout, mais il n'en reste que **un bit**.


In [9]:
-- L'orbite : re-appliquer broadcast ne change plus rien -- un point fixe en un pas.
#eval huitEtats.map (fun s =>
  let t := IIT.broadcast s;
  [t 0, t 1, t 2])
#eval huitEtats.map (fun s =>
  let t := IIT.broadcast (IIT.broadcast s);
  [t 0, t 1, t 2])


-- L'orbite : re-appliquer broadcast ne change plus rien -- un point fixe en un pas.
#eval huitEtats.map (fun s =>
─────▶  [[false, false, false],
 [true, true, true],
 [true, true, true],
 [false, false, false],
 [true, true, true],
 [false, false, false],
 [false, false, false],
 [true, true, true]]
  let t := IIT.broadcast s;
  [t 0, t 1, t 2])
#eval huitEtats.map (fun s =>
─────▶  [[false, false, false],
 [true, true, true],
 [true, true, true],
 [false, false, false],
 [true, true, true],
 [false, false, false],
 [false, false, false],
 [true, true, true]]
  let t := IIT.broadcast (IIT.broadcast s);
  [t 0, t 1, t 2])

--% env 8

Raw input:
{"cmd": "-- L'orbite : re-appliquer broadcast ne change plus rien -- un point fixe en un pas.\n#eval huitEtats.map (fun s =>\n  let t := IIT.broadcast s;\n  [t 0, t 1, t 2])\n#eval huitEtats.map (fun s =>\n  let t := IIT.broadcast (IIT.broadcast s);\n  [t 0, t 1, t 2])\n", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data":
   "[[false, false, false],\n [true, true, true],\n [true, true, true],\n [false, false, false],\n [true, true, true],\n [false, false, false],\n [false, false, false],\n [true, true, true]]"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data":
   "[[false, false, false],\n [true, true, true],\n [true, true, true],\n [false, false, false],\n [true, true, true],\n [false, false, false],\n [false, false, false],\n [true, true, true]]"}],
 "env": 8}

**Interprétation.** Les deux dernières tables sont identiques : `broadcast` atteint son point
fixe immédiatement. Le théorème du lake qui le dit en général — *toute orbite atteint un point
fixe en deux pas* (`broadcast_two_cycle`, alias figé `broadcast_trivial_dynamics`) — est la
marche 3 ; ici, il se **voit**.


## 7. Ce que la marche 2 a établi

- La parité est un **outil calculé** : récurrence tête + queue, cas `n = 0` à `false`, quatre
  lemmes relus comme des faits (tout-faux, tout-vrai, bit unique, constant).
- `broadcast` est ce seul outil transformé en système : deux images, un point fixe immédiat.
- La marche 3 assemblera les deux marches : le système qui diffuse la parité est **intégré pour
  toute coupe** (théorème B) **et** de **dynamique triviale** (théorème C) — la critique
  d'Aaronson sous forme formelle, mise en regard des mesures Φ du carnet Python `IIT-06-L`.


## 8. Exercices

Même consigne que la marche 1 : chaque stub compile tel quel (`false` ou `0`), et le `#eval` de
contrôle, à côté, montre la divergence entre votre stub et la valeur attendue — c'est le signal
à faire disparaître.


### Exercice 1 — compter les bits

Écrire `nbTrue` : le nombre de bits vrais d'un état de `n = 4`. Indice : la récurrence de
`parity` se transpose presque telle quelle, avec `( + )` au lieu de `xor`.


In [10]:
-- Exercice 1 : le nombre de bits vrais d'un etat de n = 4.
-- TODO etudiant : remplacer le corps ci-dessous.
def nbTrue (s : IIT.State 4) : Nat := 0
-- Controle : attendu [0, 1, 2, 4].
def etats4 : List (IIT.State 4) :=
  [ fun _ => false,
    fun i => decide (i.val = 0),
    fun i => decide (i.val % 2 = 0),
    fun _ => true ]
#eval etats4.map nbTrue


-- Exercice 1 : le nombre de bits vrais d'un etat de n = 4.
-- TODO etudiant : remplacer le corps ci-dessous.
def nbTrue (s : IIT.State 4) : Nat := 0
            ─▶ 🟨 Variable name `s` is not explicitly referenced.

Hint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:
  [apply] _s

Note: This linter can be disabled with `set_option linter.unusedVariables false`
-- Controle : attendu [0, 1, 2, 4].
def etats4 : List (IIT.State 4) :=
  [ fun _ => false,
    fun i => decide (i.val = 0),
    fun i => decide (i.val % 2 = 0),
    fun _ => true ]
#eval etats4.map nbTrue
─────▶  [0, 0, 0, 0]

--% env 9

Raw input:
{"cmd": "-- Exercice 1 : le nombre de bits vrais d'un etat de n = 4.\n-- TODO etudiant : remplacer le corps ci-dessous.\ndef nbTrue (s : IIT.State 4) : Nat := 0\n-- Controle : attendu [0, 1, 2, 4].\ndef etats4 : List (IIT.State 4) :=\n  [ fun _ => false,\n    fun i => decide (i.val = 0),\n    fun i => decide (i.val % 2 = 0),\n    fun _ => true ]\n#eval etats4.map nbTrue\n", "env": 8}
Raw output:
{"messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 12},
   "endPos": {"line": 3, "column": 13},
   "data":
   "Variable name `s` is not explicitly referenced.\n\nHint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:\n  [apply] _s\n\nNote: This linter can be disabled with `set_option linter.unusedVariables false`"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 5},
   "data": "[0, 0, 0, 0]"}],
 "env": 9}

### Exercice 2 — prédire le broadcast

Écrire `suivant` : l'état broadcast d'un état de `n = 3`, à la main — tous les bits égaux à la
parité de l'état.


In [11]:
-- Exercice 2 : l'etat broadcast, ecrit a la main.
-- TODO etudiant : remplacer le corps ci-dessous.
def suivant (s : IIT.State 3) : IIT.State 3 := fun _ => false
-- Controle : les deux colonnes doivent rendre la meme table.
#eval (huitEtats.map (fun s => [suivant s 0, suivant s 1, suivant s 2]),
       huitEtats.map (fun s => [IIT.broadcast s 0, IIT.broadcast s 1, IIT.broadcast s 2]))


-- Exercice 2 : l'etat broadcast, ecrit a la main.
-- TODO etudiant : remplacer le corps ci-dessous.
def suivant (s : IIT.State 3) : IIT.State 3 := fun _ => false
             ─▶ 🟨 Variable name `s` is not explicitly referenced.

Hint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:
  [apply] _s

Note: This linter can be disabled with `set_option linter.unusedVariables false`
-- Controle : les deux colonnes doivent rendre la meme table.
#eval (huitEtats.map (fun s => [suivant s 0, suivant s 1, suivant s 2]),
─────▶  ([[false, false, false],
  [false, false, false],
  [false, false, false],
  [false, false, false],
  [false, false, false],
  [false, false, false],
  [false, false, false],
  [false, false, false]],
 [[false, false, false],
  [true, true, true],
  [true, true, true],
  [false, false, false],
  [true, true, true],
  [false, false, false],
  [false, false, false],
  [true, true, true]])
       huitEtats.map (fun s => [IIT.broadcast s 0, IIT.broadcast s 1, IIT.broadcast s 2]))

--% env 10

Raw input:
{"cmd": "-- Exercice 2 : l'etat broadcast, ecrit a la main.\n-- TODO etudiant : remplacer le corps ci-dessous.\ndef suivant (s : IIT.State 3) : IIT.State 3 := fun _ => false\n-- Controle : les deux colonnes doivent rendre la meme table.\n#eval (huitEtats.map (fun s => [suivant s 0, suivant s 1, suivant s 2]),\n       huitEtats.map (fun s => [IIT.broadcast s 0, IIT.broadcast s 1, IIT.broadcast s 2]))\n", "env": 9}
Raw output:
{"messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 13},
   "endPos": {"line": 3, "column": 14},
   "data":
   "Variable name `s` is not explicitly referenced.\n\nHint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:\n  [apply] _s\n\nNote: This linter can be disabled with `set_option linter.unusedVariables false`"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data":
   "([[false, false, false],\n  [false, false, false],\n  [false, false, false],\n  [false, false, false],\n  [false, false, false],\n  [false, false, false],\n  [false, false, false],\n  [false, false, false]],\n [[false, false, false],\n  [true, true, true],\n  [true, true, true],\n  [false, false, false],\n  [true, true, true],\n  [false, false, false],\n  [false, false, false],\n  [true, true, true]])"}],
 "env": 10}

### Exercice 3 — le bit unique, n'importe où

Écrire `oneHotK` : l'état de `n` dont **seul** le bit `k` est vrai. Contrôle : pour `k = 2` et
`n = 4`, attendu `[false, false, true, false]` — et une parité vraie : seul le compte compte,
pas la position.


In [12]:
-- Exercice 3 : l'etat dont seul le bit k est vrai.
-- TODO etudiant : remplacer le corps ci-dessous.
def oneHotK {n : Nat} (k : Fin n) : IIT.State n := fun _ => false
-- Controle : attendu [false, false, true, false] puis true.
#eval [oneHotK (⟨2, by omega⟩ : Fin 4) 0, oneHotK (⟨2, by omega⟩ : Fin 4) 1,
       oneHotK (⟨2, by omega⟩ : Fin 4) 2, oneHotK (⟨2, by omega⟩ : Fin 4) 3]
#eval IIT.parity 4 (oneHotK (⟨2, by omega⟩ : Fin 4))


-- Exercice 3 : l'etat dont seul le bit k est vrai.
-- TODO etudiant : remplacer le corps ci-dessous.
def oneHotK {n : Nat} (k : Fin n) : IIT.State n := fun _ => false
                       ─▶ 🟨 Variable name `k` is not explicitly referenced.

Hint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:
  [apply] _k

Note: This linter can be disabled with `set_option linter.unusedVariables false`
-- Controle : attendu [false, false, true, false] puis true.
#eval [oneHotK (⟨2, by omega⟩ : Fin 4) 0, oneHotK (⟨2, by omega⟩ : Fin 4) 1,
─────▶  [false, false, false, false]
       oneHotK (⟨2, by omega⟩ : Fin 4) 2, oneHotK (⟨2, by omega⟩ : Fin 4) 3]
#eval IIT.parity 4 (oneHotK (⟨2, by omega⟩ : Fin 4))
─────▶  false

--% env 11

Raw input:
{"cmd": "-- Exercice 3 : l'etat dont seul le bit k est vrai.\n-- TODO etudiant : remplacer le corps ci-dessous.\ndef oneHotK {n : Nat} (k : Fin n) : IIT.State n := fun _ => false\n-- Controle : attendu [false, false, true, false] puis true.\n#eval [oneHotK (\u27e82, by omega\u27e9 : Fin 4) 0, oneHotK (\u27e82, by omega\u27e9 : Fin 4) 1,\n       oneHotK (\u27e82, by omega\u27e9 : Fin 4) 2, oneHotK (\u27e82, by omega\u27e9 : Fin 4) 3]\n#eval IIT.parity 4 (oneHotK (\u27e82, by omega\u27e9 : Fin 4))\n", "env": 10}
Raw output:
{"messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 23},
   "endPos": {"line": 3, "column": 24},
   "data":
   "Variable name `k` is not explicitly referenced.\n\nHint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:\n  [apply] _k\n\nNote: This linter can be disabled with `set_option linter.unusedVariables false`"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "[false, false, false, false]"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "false"}],
 "env": 11}

## 9. Conclusion — la parité est un outil, le système est prêt

Les déclarations 11 à 16 du lake sont maintenant exposées : une définition récursive, un
système, quatre lemmes relus comme des faits. La marche 3 n'aura plus rien à introduire — elle
pourra prouver : le broadcast-parité est **intégré pour toute coupe** et **trivial en deux pas**,
les deux sur le même système. C'est l'objet qui a mordu IIT, en théorèmes.

**Navigation** : [IIT-6 : l'objet qui a mordu IIT](./IIT-06-L-Objet-qui-a-Mordu-IIT-Python.ipynb) (moitié empirique, Φ sous PyPhi) · [IIT-7 : la coupe et le verdict](./IIT-07-Integration-Par-Coupes-Lean.ipynb)le verdict) · **ce carnet** (marche 2 : la parité) · marche 3 à venir (le capstone).
